# FineWeb-Edu Dataset Exploration

Before integrating `HuggingFaceFW/fineweb-edu` as a supplementary training corpus alongside `thekingslee/9ja-bookcorpus`, this notebook answers:

1. What fields does the dataset have?
2. What does the text actually look like?
3. What does the educational quality score mean, and what threshold should we use?
4. What are the token/length distributions?
5. How does it compare to our existing 9ja-bookcorpus?
6. How many rows do we need to hit ~300M tokens?

## 1. Setup

In [ ]:
from datasets import load_dataset
import pandas as pd
import matplotlib.pyplot as plt
from transformers import GPT2Tokenizer
import numpy as np

pd.set_option('display.max_colwidth', 300)
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
print('Setup complete.')

## 2. Load a Small Sample

We load 2,000 rows from `sample-10BT` — enough to get representative stats without hammering memory.

In [ ]:
SAMPLE_ROWS = 2_000

fw = load_dataset(
    'HuggingFaceFW/fineweb-edu',
    name='sample-10BT',
    split=f'train[:{SAMPLE_ROWS}]'
)

df = fw.to_pandas()
print(f'Loaded {len(df)} rows')
print(f'Columns: {list(df.columns)}')

## 3. Field Overview

In [ ]:
df.dtypes

In [ ]:
# Non-text columns at a glance
df[['id', 'url', 'date', 'language', 'language_score', 'token_count', 'score', 'int_score']].head(10)

## 4. Educational Quality Score

`score` (float 0–5) and `int_score` (int 0–5) are assigned by an LLM judge rating how educationally valuable each web page is:
- **0–1**: low quality (spam, ads, boilerplate)
- **2**: some educational content but mixed
- **3**: clearly educational
- **4–5**: high-quality educational content (textbooks, tutorials, academic writing)

FineWeb-Edu already pre-filters to include only pages with `int_score >= 3`, but we can still see the distribution within our slice.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df['int_score'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('int_score distribution')
axes[0].set_xlabel('Educational score (0–5)')
axes[0].set_ylabel('Count')

df['score'].plot(kind='hist', bins=40, ax=axes[1], color='steelblue')
axes[1].set_title('Raw score distribution (float)')
axes[1].set_xlabel('Score')

plt.tight_layout()
plt.show()

print(df['int_score'].value_counts().sort_index())

## 5. Sample Texts — Read the Actual Content

Most important step: read some texts at different score levels to judge quality and suitability.

In [ ]:
def show_samples(score_level, n=3):
    subset = df[df['int_score'] == score_level].head(n)
    for i, row in subset.iterrows():
        print(f'--- int_score={score_level} | url={row["url"]} ---')
        print(row['text'][:800])
        print()

for level in sorted(df['int_score'].unique()):
    show_samples(level, n=2)

## 6. Text Length & Token Count Distribution

In [ ]:
df['char_len'] = df['text'].str.len()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df['token_count'].plot(kind='hist', bins=50, ax=axes[0], color='steelblue')
axes[0].set_title('Token count per document')
axes[0].set_xlabel('Tokens')

df['char_len'].plot(kind='hist', bins=50, ax=axes[1], color='coral')
axes[1].set_title('Character length per document')
axes[1].set_xlabel('Characters')

plt.tight_layout()
plt.show()

print(df[['token_count', 'char_len']].describe().round(1))

In [ ]:
# How many rows do we need for ~300M tokens?
avg_tokens_per_row = df['token_count'].mean()
target_tokens = 300_000_000
rows_needed = int(target_tokens / avg_tokens_per_row)

print(f'Average tokens per row (this sample): {avg_tokens_per_row:.0f}')
print(f'Rows needed for 300M tokens:          {rows_needed:,}')
print(f'sample-10BT has 9.67M rows total — {rows_needed/9_670_000*100:.1f}% of it')

## 7. Language Check

All rows should be English. Verify `language` and `language_score`.

In [ ]:
print('Language distribution:')
print(df['language'].value_counts())
print(f'\nLanguage score — mean: {df["language_score"].mean():.4f}, min: {df["language_score"].min():.4f}')

## 8. Source Domain Distribution

Where does this text actually come from? Are the sources appropriate for our use case?

In [ ]:
from urllib.parse import urlparse

df['domain'] = df['url'].apply(lambda u: urlparse(u).netloc.replace('www.', ''))
top_domains = df['domain'].value_counts().head(20)

top_domains.plot(kind='barh', figsize=(10, 6), color='steelblue')
plt.title('Top 20 source domains')
plt.xlabel('Document count')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print(top_domains)

## 9. Compare With 9ja-BookCorpus

In [ ]:
# Load a sample of our own dataset for comparison
nja = load_dataset('thekingslee/9ja-bookcorpus', split='train[:2000]')
nja_df = nja.to_pandas()

print('9ja-bookcorpus columns:', list(nja_df.columns))
nja_df.head(5)

In [ ]:
# Token count comparison
nja_df['token_count'] = nja_df['text'].apply(lambda t: len(tokenizer.encode(str(t), truncation=False)))
fw_sample_tokens = df['token_count']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

nja_df['token_count'].clip(upper=1000).plot(kind='hist', bins=40, ax=axes[0], color='coral')
axes[0].set_title('9ja-bookcorpus token counts')
axes[0].set_xlabel('Tokens (clipped at 1000)')

fw_sample_tokens.clip(upper=1000).plot(kind='hist', bins=40, ax=axes[1], color='steelblue')
axes[1].set_title('FineWeb-Edu token counts')
axes[1].set_xlabel('Tokens (clipped at 1000)')

plt.tight_layout()
plt.show()

print(f'9ja-bookcorpus — avg tokens: {nja_df["token_count"].mean():.0f}, median: {nja_df["token_count"].median():.0f}')
print(f'FineWeb-Edu    — avg tokens: {fw_sample_tokens.mean():.0f}, median: {fw_sample_tokens.median():.0f}')

In [ ]:
# Read some 9ja texts for comparison
print('=== 9ja-bookcorpus samples ===')
for _, row in nja_df.head(3).iterrows():
    print(str(row['text'])[:500])
    print()

## 10. Score Threshold Decision

Should we use all rows (int_score >= 3, pre-filtered by FineWeb) or restrict further to int_score >= 4?
Higher threshold = smaller but cleaner dataset.

In [ ]:
for threshold in [3, 4, 5]:
    subset = df[df['int_score'] >= threshold]
    pct = len(subset) / len(df) * 100
    avg_tok = subset['token_count'].mean()
    # Extrapolate: at this threshold, how many rows from 9.67M to hit 300M tokens?
    rows_needed_thresh = int(300_000_000 / avg_tok)
    print(f'int_score >= {threshold}: {len(subset):,} rows ({pct:.0f}% of sample) | '
          f'avg {avg_tok:.0f} tokens | need ~{rows_needed_thresh:,} rows for 300M tokens')

## 11. Summary & Decision

Based on the analysis above, note:

- **Score distribution**: What % is int_score 3 vs 4 vs 5?
- **Source domains**: Educational (.edu), Wikipedia, news, documentation?
- **Text quality**: Does the content look clean and suitable for language modeling?
- **Token estimate**: How many rows needed for 300M tokens at each score threshold?

Recommended next step: use `int_score >= 3` (the full FineWeb-Edu pre-filter) with ~300K rows sliced via `split='train[:300000]'` in `book_corpus.py`.

---
## 12. Prepare & Save 300M Tokens to Google Drive

Run this section **once in Google Colab** to produce a pre-tokenized binary file on Drive (~600 MB).
Subsequent training runs load directly from this file — no re-downloading, no re-tokenizing every run.

| Step | What happens |
|------|--------------|
| 12a | Mount Google Drive |
| 12b | Load 300 000 rows from `sample-10BT` (~310M tokens) |
| 12c | Batch-tokenize with GPT-2 tokenizer |
| 12d | Save as compact `uint16` numpy array to Drive |
| 12e | Verify: reload and spot-check decoded text |

> **Why uint16?** GPT-2 vocab is 50 257 — fits in uint16 (max 65 535).
> 300M tokens x 2 bytes = **~600 MB** on Drive.

### 12a — Mount Google Drive

Skip if already mounted.

In [ ]:
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/GPT1_Data'
    print('Running in Colab — saving to Google Drive')
except ModuleNotFoundError:
    # Running locally — save next to this notebook instead
    SAVE_DIR = os.path.join(os.getcwd(), 'fineweb_tokens')
    print('Running locally — saving to:', SAVE_DIR)

os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Save directory ready: {SAVE_DIR}')

### 12b — Load 300 000 rows from FineWeb-Edu

At ~1 034 tokens/row (verified in Section 6 above), 300 000 rows gives ~310M tokens.
Adjust `FINEWEB_ROWS` to hit your exact target.

In [ ]:
from datasets import load_dataset
import pandas as pd
import matplotlib.pyplot as plt
from transformers import GPT2Tokenizer
import numpy as np
FINEWEB_ROWS = 300_000

print(f'Loading {FINEWEB_ROWS:,} rows ...')
fw_full = load_dataset(
    'HuggingFaceFW/fineweb-edu',
    name='sample-10BT',
    split=f'train[:{FINEWEB_ROWS}]'
)
print(f'Loaded {len(fw_full):,} rows')
print(f'Sample: {fw_full[0]["text"][:200]}')

### 12c — Batch Tokenize

Tokenizes in batches of 1 000 rows — roughly 10-20x faster than calling
`tokenizer.encode()` one text at a time (the current `book_corpus.py` approach).
Each text is truncated at `MAX_LEN` tokens.

In [ ]:
fw_full.save_to_disk(os.path.join(SAVE_DIR, 'fineweb_300k'))
print('Saved.')


In [ ]:
fw_full.to_parquet('fineweb_300k.parquet')
print(f'Saved. Size: {os.path.getsize("fineweb_300k.parquet")/1e6:.1f} MB')


In [ ]:
import os
from huggingface_hub import HfApi

api = HfApi(token="os.environ.get("HF_TOKEN")")   # get from huggingface.co/settings/tokens

api.create_repo(repo_id='Asharox/fineweb-300k', repo_type='dataset', exist_ok=True)


api.upload_file(
    path_or_fileobj='fineweb_300k.parquet',
    path_in_repo='fineweb_300k.parquet',
    repo_id='Asharox/fineweb-300k',    # change to your HF username/repo-name
    repo_type='dataset',
)
print('Uploaded.')


In [ ]:
from datasets import load_dataset
fw_full = load_dataset('thekingslee/fineweb-300k', split='train')
